In [1]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -qr requirements.txt
import torch
from IPython.display import Image, clear_output

clear_output()
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

Setup complete. Using torch 2.10.0+cu128 (Tesla T4)


In [2]:
import os
import xml.etree.ElementTree as ET
from tqdm import tqdm
import PIL.Image
import requests
import tarfile

# 1. Download and Extract VOC2012
url = "http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar"
if not os.path.exists('VOCtrainval_11-May-2012.tar'):
    open('VOCtrainval_11-May-2012.tar', 'wb').write(requests.get(url).content)
    with tarfile.open('VOCtrainval_11-May-2012.tar') as tar:
        tar.extractall(path='../')

# 2. Define Classes
classes = ["aeroplane", "bicycle", "bird", "boat", "bottle", "bus", "car", "cat", "chair", "cow", "diningtable", "dog", "horse", "motorbike", "person", "pottedplant", "sheep", "sofa", "train", "tvmonitor"]

def convert(size, box):
    dw = 1./size[0]
    dh = 1./size[1]
    x = (box[0] + box[1])/2.0
    y = (box[2] + box[3])/2.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x*dw, y*dh, w*dw, h*dh)

def process_voc_data(xml_path, output_path, image_dir):
    os.makedirs(output_path, exist_ok=True)
    for xml_file in tqdm(os.listdir(xml_path)):
        tree = ET.parse(os.path.join(xml_path, xml_file))
        root = tree.getroot()
        img_name = root.find('filename').text
        img_p = os.path.join(image_dir, img_name)
        
        # CRITICAL: RGB/Grayscale/RGBA Check
        try:
            with PIL.Image.open(img_p) as img:
                if img.mode != 'RGB':
                    img = img.convert('RGB')
                    img.save(img_p) 
        except Exception as e:
            print(f"Skipping {img_name}: {e}")
            continue

        size = root.find('size')
        w = int(size.find('width').text)
        h = int(size.find('height').text)

        with open(os.path.join(output_path, img_name.replace('.jpg', '.txt')), 'w') as f:
            for obj in root.iter('object'):
                cls = obj.find('name').text
                if cls not in classes: continue
                cls_id = classes.index(cls)
                xmlbox = obj.find('bndbox')
                b = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text), 
                     float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))
                bb = convert((w, h), b)
                f.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}\n")

# Run Conversion
process_voc_data('../VOCdevkit/VOC2012/Annotations', '../VOCdevkit/VOC2012/labels', '../VOCdevkit/VOC2012/JPEGImages')

# Create Train/Val Split (Simple 90/10 for research)
images = [f for f in os.listdir('../VOCdevkit/VOC2012/JPEGImages') if f.endswith('.jpg')]
with open('../train.txt', 'w') as f:
    for img in images[:int(len(images)*0.9)]: f.write(f"../VOCdevkit/VOC2012/JPEGImages/{img}\n")
with open('../val.txt', 'w') as f:
    for img in images[int(len(images)*0.9):]: f.write(f"../VOCdevkit/VOC2012/JPEGImages/{img}\n")

/tmp/ipykernel_24/73529266.py:13: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path='../')
100%|██████████| 17125/17125 [00:04<00:00, 3655.31it/s]


In [3]:
import yaml

data = {
    'train': '../train.txt',
    'val': '../val.txt',
    'nc': 20,
    'names': classes
}

with open('voc_data.yaml', 'w') as f:
    yaml.dump(data, f)

In [4]:
import os

# 1. Rename the images directory so YOLOv5's internal path replacement works
if os.path.exists('../VOCdevkit/VOC2012/JPEGImages'):
    os.rename('../VOCdevkit/VOC2012/JPEGImages', '../VOCdevkit/VOC2012/images')

# 2. Re-write train.txt and val.txt with the correct 'images' folder name
images = [f for f in os.listdir('../VOCdevkit/VOC2012/images') if f.endswith('.jpg')]

with open('../train.txt', 'w') as f:
    for img in images[:int(len(images)*0.9)]: 
        f.write(f"../VOCdevkit/VOC2012/images/{img}\n")
        
with open('../val.txt', 'w') as f:
    for img in images[int(len(images)*0.9):]: 
        f.write(f"../VOCdevkit/VOC2012/images/{img}\n")

# 3. Fix the Albumentations warning from your logs
!pip install "albumentations<1.4.0" -q

print("✅ Directories fixed! Ready for training.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.7/125.7 kB 5.4 MB/s eta 0:00:00
✅ Directories fixed! Ready for training.


In [5]:
# WANDB_MODE=disabled prevents the interactive prompt from pausing your training
# YOLOv5 will still save all your metrics locally to runs/train/exp/results.csv

!WANDB_MODE=disabled python train.py \
    --img 640 \
    --batch 32 \
    --epochs 50 \
    --data voc_data.yaml \
    --weights yolov5s.pt \
    --device 0,1

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
wandb: WARNING ⚠️ wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
2026-03-27 08:29:27.351473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774600167.533106      69 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774600167.586030      69 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS wh

In [6]:
import torch
import time
import pandas as pd
import glob
import os

print("========== YOLOv5 RESEARCH METRICS ==========\n")

# FAIL-SAFE 1: Dynamically find the results.csv and best.pt
# Even if YOLOv5 names the folder exp2 or exp3, this will find it.
csv_files = glob.glob('/kaggle/working/yolov5/runs/train/*/results.csv')
weight_files = glob.glob('/kaggle/working/yolov5/runs/train/*/weights/best.pt')

if csv_files:
    latest_csv = max(csv_files, key=os.path.getctime)
    df = pd.read_csv(latest_csv)
    df.columns = df.columns.str.strip() 
    best_epoch = df.iloc[df['metrics/mAP_0.5:0.95'].idxmax()]
    print("--- Accuracy (Best Epoch) ---")
    print(f"mAP@50:      {best_epoch['metrics/mAP_0.5']:.4f}")
    print(f"mAP@50-95:   {best_epoch['metrics/mAP_0.5:0.95']:.4f}\n")
else:
    print("⚠️ results.csv not found in this cell, but will be zipped in Cell 6.\n")

# FAIL-SAFE 2: Try/Except block for benchmarking. 
# If this fails, it will NOT crash the notebook. It will proceed to zip your files.
try:
    print("--- Speed (Inference Only) ---")
    if weight_files:
        latest_weights = max(weight_files, key=os.path.getctime)
        device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        
        # Load model using local source
        model = torch.hub.load('/kaggle/working/yolov5', 'custom', path=latest_weights, source='local', force_reload=True)
        model.to(device)
        model.eval()

        dummy_input = torch.randn(1, 3, 640, 640).to(device)

        # Warmup
        for _ in range(50): _ = model(dummy_input)

        # Benchmark
        iterations = 500
        start_time = time.time()
        with torch.no_grad():
            for _ in range(iterations): _ = model(dummy_input)
        end_time = time.time()

        latency_ms = ((end_time - start_time) / iterations) * 1000
        print(f"Hardware:    {torch.cuda.get_device_name(0)}")
        print(f"Latency:     {latency_ms:.2f} ms per image")
        print(f"Throughput:  {(1000 / latency_ms):.2f} FPS")
    else:
        print("⚠️ best.pt not found for benchmarking.")
except Exception as e:
    print(f"⚠️ Non-fatal benchmarking error. Proceeding to zip files. Error: {e}")

========== YOLOv5 RESEARCH METRICS ==========

--- Accuracy (Best Epoch) ---
mAP@50:      0.6864
mAP@50-95:   0.4752

--- Speed (Inference Only) ---


YOLOv5 🚀 v7.0-463-g88af13e3 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Fusing layers... 
Model summary: 157 layers, 7064065 parameters, 0 gradients, 15.9 GFLOPs
Adding AutoShape... 
/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/k

Hardware:    Tesla T4
Latency:     7.01 ms per image
Throughput:  142.58 FPS


/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/kaggle/working/yolov5/models/common.py:872: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
/kagg

In [7]:
import os

# Ensure we are in the root working directory
os.chdir('/kaggle/working')

# Zip the ENTIRE runs directory. 
# This guarantees that no matter what YOLOv5 names the output folder, it gets saved.
!zip -r yolov5_voc_results.zip /kaggle/working/yolov5/runs/

print("\n✅ Run completely finished! Archive 'yolov5_voc_results.zip' is secured.")

  adding: kaggle/working/yolov5/runs/ (stored 0%)
  adding: kaggle/working/yolov5/runs/train/ (stored 0%)
  adding: kaggle/working/yolov5/runs/train/exp/ (stored 0%)
  adding: kaggle/working/yolov5/runs/train/exp/hyp.yaml (deflated 45%)
  adding: kaggle/working/yolov5/runs/train/exp/events.out.tfevents.1774600184.b09257239798.69.0 (deflated 92%)
  adding: kaggle/working/yolov5/runs/train/exp/labels_correlogram.jpg (deflated 33%)
  adding: kaggle/working/yolov5/runs/train/exp/train_batch0.jpg (deflated 4%)
  adding: kaggle/working/yolov5/runs/train/exp/results.csv (deflated 83%)
  adding: kaggle/working/yolov5/runs/train/exp/train_batch1.jpg (deflated 4%)
  adding: kaggle/working/yolov5/runs/train/exp/train_batch2.jpg (deflated 3%)
  adding: kaggle/working/yolov5/runs/train/exp/opt.yaml (deflated 50%)
  adding: kaggle/working/yolov5/runs/train/exp/labels.jpg (deflated 15%)
  adding: kaggle/working/yolov5/runs/train/exp/weights/ (stored 0%)
  adding: kaggle/working/yolov5/runs/train/exp/